In [6]:
# ═══════════════════════════════════════════════════════════
# 0) reco DM 로드 + _1 / _2 서브셋 슬라이스
# ═══════════════════════════════════════════════════════════
reco_dm = events["hltHpsPFTau_decayMode"].array()   # branch 이름이 다르면 여기서 바꿔

reco_dm_1 = reco_dm[trigger_passed][mask2]
reco_dm_2 = reco_dm[trigger_passed][mask3]


# ═══════════════════════════════════════════════════════════
# 1) 2D (gen DM × reco DM) 플롯 helper
# ═══════════════════════════════════════════════════════════
DM_VALS = [-1, 0, 1, 2, 5, 10, 11, 15]   # -1 = reco 없음/이상치
DM_LBLS = [str(v) for v in DM_VALS]

def plot_gen_reco_dm_2d(gen_dm, reco_dm, title, save_path=None):
    gen_dm  = np.asarray(gen_dm,  dtype=int)
    reco_dm = np.asarray(reco_dm, dtype=int)
    n = len(DM_VALS)
    M = np.zeros((n, n), dtype=int)   # rows = gen, cols = reco
    for i, g in enumerate(DM_VALS):
        for j, r in enumerate(DM_VALS):
            M[i, j] = int(np.sum((gen_dm == g) & (reco_dm == r)))

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(M, origin='lower', cmap='viridis', aspect='auto')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(DM_LBLS); ax.set_yticklabels(DM_LBLS)
    ax.set_xlabel('Reco DM (closest reco)')
    ax.set_ylabel('Gen DM (GenVisTau_status)')
    ax.set_title(f'{title}\nN = {len(gen_dm)}')
    cbar = plt.colorbar(im, ax=ax); cbar.set_label('Counts')

    vmax = M.max() if M.max() > 0 else 1
    for i in range(n):
        for j in range(n):
            if M[i, j] > 0:
                ax.text(j, i, str(M[i, j]),
                        ha='center', va='center', fontsize=9,
                        color='white' if M[i, j] < vmax * 0.5 else 'black')
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches='tight')
        plt.close(fig)
    else:
        plt.show()


# ═══════════════════════════════════════════════════════════
# 2) analyze_tau_matching  (reco_dm_sel 추가 + 분기별 2D 플롯)
# ═══════════════════════════════════════════════════════════
def analyze_tau_matching(
    reco_pt_sel, reco_eta_sel, reco_phi_sel, reco_dm_sel,
    gen_pt_sel,  gen_eta_sel,  gen_phi_sel,  gen_dm_sel,
    njets_sel,
    DR_CUT=0.3, label="",
    gen_pt_cut=130, gen_eta_cut=2.1, reco_pt_cut=130,
    save_dir=None,
):
    # ── 매칭 로직 (그대로) ────────────────────────────────
    gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(gen_eta_sel[:, np.newaxis], reco_eta_sel)
    gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(gen_phi_sel[:, np.newaxis], reco_phi_sel)
    deta = gen_eta_bc - reco_eta_bc
    dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
    deltaR = np.sqrt(deta**2 + dphi**2)

    best_dR  = ak.min(deltaR, axis=1)
    best_idx = ak.argmin(deltaR, axis=1, keepdims=True)
    matched_mask = best_dR < DR_CUT

    matched_reco_pt  = ak.flatten(reco_pt_sel[matched_mask][best_idx[matched_mask]])
    matched_gen_pt   = gen_pt_sel[matched_mask]
    matched_gen_eta  = gen_eta_sel[matched_mask]
    matched_gen_dm   = gen_dm_sel[matched_mask]
    matched_best_dR  = best_dR[matched_mask]

    pt_eta_cut = (matched_gen_pt > gen_pt_cut) & (np.abs(matched_gen_eta) < gen_eta_cut)
    after_gen_sel_gen_pt  = matched_gen_pt[pt_eta_cut]
    after_gen_sel_reco_pt = matched_reco_pt[pt_eta_cut]
    after_gen_sel_dr      = matched_best_dR[pt_eta_cut]

    reco_pt_after_cut = reco_pt_sel[matched_mask][pt_eta_cut]
    sorted_idx   = ak.argsort(reco_pt_after_cut, axis=1, ascending=False)
    leading_reco = reco_pt_after_cut[sorted_idx][:, 0]

    pt130_filter = after_gen_sel_reco_pt > reco_pt_cut
    lead_reco_pass    = leading_reco[pt130_filter]
    matched_reco_pass = after_gen_sel_reco_pt[pt130_filter]
    has_higher        = lead_reco_pass > matched_reco_pass

    lead_reco_fail    = leading_reco[~pt130_filter]
    matched_reco_fail = after_gen_sel_reco_pt[~pt130_filter]
    has_higher_fail   = lead_reco_fail > matched_reco_fail

    # ── numpy 변환 + 분기별 DM/njets/reco_dm 준비 ─────────
    def _tonp_bool(x):
        return ak.to_numpy(ak.fill_none(x, False)).astype(bool)

    # closest reco DM (매칭 실패 이벤트에도 "그나마 가까웠던" reco의 DM 을 씀)
    closest_reco_dm = ak.to_numpy(ak.fill_none(
        ak.flatten(reco_dm_sel[best_idx]), -1
    )).astype(int)

    njets_sel_np = np.asarray(njets_sel, dtype=int)
    dm_sel_np    = ak.to_numpy(gen_dm_sel).astype(int)

    mm_np          = _tonp_bool(matched_mask)
    dm_matched     = dm_sel_np[mm_np]
    reco_dm_match  = closest_reco_dm[mm_np]
    njets_matched  = njets_sel_np[mm_np]

    dm_unmatched    = dm_sel_np[~mm_np]
    reco_dm_unmatch = closest_reco_dm[~mm_np]

    pe_np             = _tonp_bool(pt_eta_cut)
    dm_after_cut      = dm_matched[pe_np]
    reco_dm_after_cut = reco_dm_match[pe_np]
    njets_after_cut   = njets_matched[pe_np]

    p130_np           = _tonp_bool(pt130_filter)
    dm_p130_pass      = dm_after_cut[p130_np]
    reco_dm_p130_pass = reco_dm_after_cut[p130_np]
    njets_p130_pass   = njets_after_cut[p130_np]
    dm_p130_fail      = dm_after_cut[~p130_np]
    reco_dm_p130_fail = reco_dm_after_cut[~p130_np]
    njets_p130_fail   = njets_after_cut[~p130_np]

    hh_np                  = _tonp_bool(has_higher)
    hhf_np                 = _tonp_bool(has_higher_fail)
    dm_lead_higher         = dm_p130_pass[hh_np]
    reco_dm_lead_higher    = reco_dm_p130_pass[hh_np]
    njets_lead_higher      = njets_p130_pass[hh_np]
    dm_matched_lead        = dm_p130_pass[~hh_np]
    reco_dm_matched_lead   = reco_dm_p130_pass[~hh_np]
    njets_matched_lead     = njets_p130_pass[~hh_np]
    dm_fail_higher         = dm_p130_fail[hhf_np]
    reco_dm_fail_higher    = reco_dm_p130_fail[hhf_np]
    njets_fail_higher      = njets_p130_fail[hhf_np]

    # ── summary + DM×Jet 표 + 2D 플롯 ─────────────────────
    print(f"\n{'='*60}\n[{label}]  DR_CUT = {DR_CUT}\n{'='*60}")
    print(f"  입력 이벤트 수                          : {len(gen_pt_sel)}")

    def _plot(tag, gen_d, reco_d):
        sp = None
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            safe = tag.replace(' ', '_').replace('>', 'gt').replace('<', 'lt').replace('/', '_')
            sp = os.path.join(save_dir, f"{label.replace(' ','_')}_DR{DR_CUT}_{safe}.png")
        plot_gen_reco_dm_2d(gen_d, reco_d, f"[{label}] {tag}  (DR<{DR_CUT})", save_path=sp)

    # 1) 매칭 성공
    print(f"  deltaR < {DR_CUT} 매칭 성공               : {ak.sum(matched_mask)}")
    print_dm_jet_table("매칭 성공", dm_matched, njets_matched, indent="    ")
    _plot("matched",            dm_matched,    reco_dm_match)

    # 2) 매칭 실패  ← "gen 이 reco 로 뭘로 잘못됐는지"
    print(f"  매칭 실패                               : {ak.sum(~matched_mask)}")
    _plot("unmatched",          dm_unmatched,  reco_dm_unmatch)

    # 3) gen pt/eta cut 통과
    print(f"  gen pt>{gen_pt_cut}, |eta|<{gen_eta_cut}          : {len(after_gen_sel_gen_pt)}")
    print_dm_jet_table("gen pt/eta cut 통과", dm_after_cut, njets_after_cut, indent="    ")
    _plot("after_gen_ptEta",    dm_after_cut,  reco_dm_after_cut)

    # 4) matched reco pt > 130
    print(f"  └─ matched reco pt > {reco_pt_cut}          : {len(matched_reco_pass)}")
    print_dm_jet_table(f"matched reco pt > {reco_pt_cut}",
                       dm_p130_pass, njets_p130_pass, indent="      ")
    _plot(f"recoPt_gt{reco_pt_cut}", dm_p130_pass, reco_dm_p130_pass)

    # 5) leading reco > matched (unknown)
    print(f"     ├─ leading reco > matched (unknown)  : {int(ak.sum(has_higher))}")
    print_dm_jet_table("leading reco > matched (unknown)",
                       dm_lead_higher, njets_lead_higher, indent="        ")
    _plot("lead_gt_matched",    dm_lead_higher,  reco_dm_lead_higher)

    # 6) matched is leading (triggered)
    print(f"     └─ matched is leading (triggered)    : {len(matched_reco_pass) - int(ak.sum(has_higher))}")
    print_dm_jet_table("matched is leading (triggered)",
                       dm_matched_lead, njets_matched_lead, indent="        ")
    _plot("matched_is_leading", dm_matched_lead, reco_dm_matched_lead)

    # 7) matched reco pt < 130
    print(f"  └─ matched reco pt < {reco_pt_cut}          : {len(matched_reco_fail)}")
    print_dm_jet_table(f"matched reco pt < {reco_pt_cut}",
                       dm_p130_fail, njets_p130_fail, indent="      ")
    _plot(f"recoPt_lt{reco_pt_cut}", dm_p130_fail, reco_dm_p130_fail)

    # 8) has higher reco than matched (in fail)
    print(f"     └─ has higher reco than matched      : {int(ak.sum(has_higher_fail))}")
    print_dm_jet_table("has higher reco than matched",
                       dm_fail_higher, njets_fail_higher, indent="        ")
    _plot("fail_has_higher",    dm_fail_higher, reco_dm_fail_higher)

    return {
        "matched_mask"          : matched_mask,
        "matched_gen_pt"        : matched_gen_pt,
        "matched_gen_eta"       : matched_gen_eta,
        "matched_gen_dm"        : matched_gen_dm,
        "matched_reco_pt"       : matched_reco_pt,
        "matched_best_dR"       : matched_best_dR,
        "after_gen_sel_gen_pt"  : after_gen_sel_gen_pt,
        "after_gen_sel_reco_pt" : after_gen_sel_reco_pt,
        "after_gen_sel_dr"      : after_gen_sel_dr,
        "leading_reco"          : leading_reco,
    }


# ═══════════════════════════════════════════════════════════
# 3) scan_dr_gen1 / scan_dr_gen2  (reco_dm_1/2 전달)
# ═══════════════════════════════════════════════════════════
def scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1, reco_dm_1,
                 gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                 njets_1,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5],
                 save_dir=None):
    gen_pt_flat  = ak.flatten(gen_pt_1)
    gen_eta_flat = ak.flatten(gen_eta_1)
    gen_phi_flat = ak.flatten(gen_phi_1)
    gen_dm_flat  = ak.flatten(gen_dm_1)
    njets_arr    = np.asarray(njets_1, dtype=int)

    results = {}
    for dr in dr_list:
        results[dr] = analyze_tau_matching(
            reco_pt_1, reco_eta_1, reco_phi_1, reco_dm_1,
            gen_pt_flat, gen_eta_flat, gen_phi_flat, gen_dm_flat,
            njets_arr,
            DR_CUT=dr, label="Gen=1",
            save_dir=save_dir,
        )
    return results


def scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2, reco_dm_2,
                 gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                 njets_2,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5],
                 save_dir=None):
    gen_sort_idx   = ak.argsort(gen_pt_2, axis=1, ascending=False)
    gen_pt_sorted  = gen_pt_2[gen_sort_idx]
    gen_eta_sorted = gen_eta_2[gen_sort_idx]
    gen_phi_sorted = gen_phi_2[gen_sort_idx]
    gen_dm_sorted  = gen_dm_2[gen_sort_idx]

    lead_pt  = ak.flatten(gen_pt_sorted[:, 0:1])
    lead_eta = ak.flatten(gen_eta_sorted[:, 0:1])
    lead_phi = ak.flatten(gen_phi_sorted[:, 0:1])
    lead_dm  = ak.flatten(gen_dm_sorted[:, 0:1])
    sub_pt   = ak.flatten(gen_pt_sorted[:, 1:2])
    sub_eta  = ak.flatten(gen_eta_sorted[:, 1:2])
    sub_phi  = ak.flatten(gen_phi_sorted[:, 1:2])
    sub_dm   = ak.flatten(gen_dm_sorted[:, 1:2])

    njets_arr = np.asarray(njets_2, dtype=int)

    results = {}
    for dr in dr_list:
        print(f"\n{'#'*60}\n  Gen=2  |  DR_CUT = {dr}\n{'#'*60}")

        gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(lead_eta[:, np.newaxis], reco_eta_2)
        gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(lead_phi[:, np.newaxis], reco_phi_2)
        deta = gen_eta_bc - reco_eta_bc
        dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
        deltaR_lead  = np.sqrt(deta**2 + dphi**2)
        best_dR_lead = ak.min(deltaR_lead, axis=1)

        lead_matched_mask = best_dR_lead < dr
        lead_fail_mask    = ~lead_matched_mask
        lead_matched_np   = ak.to_numpy(ak.fill_none(lead_matched_mask, False)).astype(bool)
        lead_fail_np      = ~lead_matched_np

        print(f"\n  [Leading gen matched: {ak.sum(lead_matched_mask)} / {len(lead_pt)} events]")
        res_lead = analyze_tau_matching(
            reco_pt_2[lead_matched_mask], reco_eta_2[lead_matched_mask],
            reco_phi_2[lead_matched_mask], reco_dm_2[lead_matched_mask],
            lead_pt[lead_matched_mask],   lead_eta[lead_matched_mask],
            lead_phi[lead_matched_mask],  lead_dm[lead_matched_mask],
            njets_arr[lead_matched_np],
            DR_CUT=dr, label="Gen2_LeadMatched", save_dir=save_dir,
        )

        print(f"\n  [Leading 매칭 실패 → Subleading 시도: {ak.sum(lead_fail_mask)} events]")
        res_sub = analyze_tau_matching(
            reco_pt_2[lead_fail_mask], reco_eta_2[lead_fail_mask],
            reco_phi_2[lead_fail_mask], reco_dm_2[lead_fail_mask],
            sub_pt[lead_fail_mask],   sub_eta[lead_fail_mask],
            sub_phi[lead_fail_mask],  sub_dm[lead_fail_mask],
            njets_arr[lead_fail_np],
            DR_CUT=dr, label="Gen2_SubLeading", save_dir=save_dir,
        )
        results[dr] = {"leading": res_lead, "subleading": res_sub}
    return results


# ═══════════════════════════════════════════════════════════
# 4) 실행
# ═══════════════════════════════════════════════════════════
SAVE_DIR = "./dm_2d_plots"   # None 으로 두면 inline 으로 show

dr_list = [0.1]

results_gen1 = scan_dr_gen1(
    reco_pt_1, reco_eta_1, reco_phi_1, reco_dm_1,
    gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
    njets_1, dr_list, save_dir=SAVE_DIR,
)

results_gen2 = scan_dr_gen2(
    reco_pt_2, reco_eta_2, reco_phi_2, reco_dm_2,
    gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
    njets_2, dr_list, save_dir=SAVE_DIR,
)

NameError: name 'trigger_passed' is not defined

In [1]:
import numpy as np
import awkward as ak

# ═══════════════════════════════════════════════════════════
# 1. DM / Jet 그룹 정의
# ═══════════════════════════════════════════════════════════
DM_GROUPS = {
    "1Prong_DM0":  [0],
    "1Prong_DM1":  [1],
    "1Prong_DM2":  [2],
    "3Prong_DM10": [10],
    "3Prong_DM11": [11],
    "1Prong_all":  [0, 1, 2],
    "3Prong_all":  [10, 11],
}
JET_GROUPS = {
    "0jet":   lambda n: n == 0,
    "1jet":   lambda n: n == 1,
    ">=2jet": lambda n: n >= 2,
}

# ═══════════════════════════════════════════════════════════
# 2. 헬퍼 함수
# ═══════════════════════════════════════════════════════════
def _dR(eta1, phi1, eta2, phi2):
    deta = eta1 - eta2
    dphi = (phi1 - phi2 + np.pi) % (2 * np.pi) - np.pi
    return np.sqrt(deta**2 + dphi**2)


def count_isolated_jets(genvis_eta, genvis_phi,
                        genjet_eta, genjet_phi,
                        deltaR_threshold=0.3):
    """이벤트별, GenVisTau 어느 것과도 ΔR ≥ threshold 인 GenJet 수 (N,) 반환"""
    n_arr = []
    for i in range(len(genvis_eta)):
        g_eta_i = np.array(genvis_eta[i])
        g_phi_i = np.array(genvis_phi[i])
        jet_eta_i = np.array(genjet_eta[i])
        jet_phi_i = np.array(genjet_phi[i])
        count = 0
        for j in range(len(jet_eta_i)):
            j_eta = float(jet_eta_i[j]); j_phi = float(jet_phi_i[j])
            is_iso = all(
                _dR(float(g_eta_i[t]), float(g_phi_i[t]), j_eta, j_phi) >= deltaR_threshold
                for t in range(len(g_eta_i))
            )
            if is_iso:
                count += 1
        n_arr.append(count)
    return np.array(n_arr, dtype=int)


def print_dm_jet_table(label, dm_arr, njets_arr, indent="    "):
    dm_arr    = np.asarray(dm_arr,    dtype=int)
    njets_arr = np.asarray(njets_arr, dtype=int)
    total_n   = len(dm_arr)
    jet_keys  = list(JET_GROUPS.keys())
    col_w     = 12

    print(f"{indent}[{label}]  subset N = {total_n}")
    header = f"{indent}{'DM':<15}"
    for jk in jet_keys:
        header += f"{jk:>{col_w}}"
    header += f"{'total':>{col_w}}"
    print(header)
    print(f"{indent}{'-' * (15 + col_w * (len(jet_keys) + 1))}")

    if total_n == 0:
        print(f"{indent}(empty)")
        return

    for dk, modes in DM_GROUPS.items():
        dm_mask = np.zeros(total_n, dtype=bool)
        for m in modes:
            dm_mask |= (dm_arr == m)
        row = f"{indent}{dk:<15}"
        total = 0
        for jk in jet_keys:
            sel = JET_GROUPS[jk]
            cnt = int(np.sum(dm_mask & sel(njets_arr)))
            row += f"{cnt:>{col_w}}"
            total += cnt
        row += f"{total:>{col_w}}"
        print(row)


# ═══════════════════════════════════════════════════════════
# 3. 매칭 분석 (njets 포함, DM×Jet 표 출력)
# ═══════════════════════════════════════════════════════════
def analyze_tau_matching(
    reco_pt_sel, reco_eta_sel, reco_phi_sel,
    gen_pt_sel,  gen_eta_sel,  gen_phi_sel,  gen_dm_sel,
    njets_sel,
    DR_CUT=0.3,
    label="",
    gen_pt_cut=130, gen_eta_cut=2.1, reco_pt_cut=130,
):
    # ---- ΔR 매칭 ----
    gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(gen_eta_sel[:, np.newaxis], reco_eta_sel)
    gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(gen_phi_sel[:, np.newaxis], reco_phi_sel)
    deta = gen_eta_bc - reco_eta_bc
    dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
    deltaR = np.sqrt(deta**2 + dphi**2)

    best_dR  = ak.min(deltaR, axis=1)
    best_idx = ak.argmin(deltaR, axis=1, keepdims=True)
    matched_mask = best_dR < DR_CUT

    matched_reco_pt  = ak.flatten(reco_pt_sel[matched_mask][best_idx[matched_mask]])
    matched_gen_pt   = gen_pt_sel[matched_mask]
    matched_gen_eta  = gen_eta_sel[matched_mask]
    matched_gen_dm   = gen_dm_sel[matched_mask]
    matched_best_dR  = best_dR[matched_mask]

    # ---- gen pt/η cut ----
    pt_eta_cut = (matched_gen_pt > gen_pt_cut) & (np.abs(matched_gen_eta) < gen_eta_cut)
    after_gen_sel_gen_pt  = matched_gen_pt[pt_eta_cut]
    after_gen_sel_reco_pt = matched_reco_pt[pt_eta_cut]
    after_gen_sel_dr      = matched_best_dR[pt_eta_cut]

    reco_pt_after_cut = reco_pt_sel[matched_mask][pt_eta_cut]
    sorted_idx        = ak.argsort(reco_pt_after_cut, axis=1, ascending=False)
    leading_reco      = reco_pt_after_cut[sorted_idx][:, 0]

    # ---- L1 cut: matched reco pt > 130 ----
    pt130_filter = after_gen_sel_reco_pt > reco_pt_cut
    lead_reco_pass    = leading_reco[pt130_filter]
    matched_reco_pass = after_gen_sel_reco_pt[pt130_filter]
    has_higher        = lead_reco_pass > matched_reco_pass

    lead_reco_fail    = leading_reco[~pt130_filter]
    matched_reco_fail = after_gen_sel_reco_pt[~pt130_filter]
    has_higher_fail   = lead_reco_fail > matched_reco_fail

    # ---- DM/njets numpy array 정리 ----
    def _tonp_bool(x):
        return ak.to_numpy(ak.fill_none(x, False)).astype(bool)

    njets_sel_np = np.asarray(njets_sel, dtype=int)
    dm_sel_np    = ak.to_numpy(gen_dm_sel).astype(int)

    mm_np            = _tonp_bool(matched_mask)
    dm_matched       = dm_sel_np[mm_np]
    njets_matched    = njets_sel_np[mm_np]

    pe_np            = _tonp_bool(pt_eta_cut)
    dm_after_cut     = dm_matched[pe_np]
    njets_after_cut  = njets_matched[pe_np]

    p130_np          = _tonp_bool(pt130_filter)
    dm_p130_pass     = dm_after_cut[p130_np]
    njets_p130_pass  = njets_after_cut[p130_np]
    dm_p130_fail     = dm_after_cut[~p130_np]
    njets_p130_fail  = njets_after_cut[~p130_np]

    hh_np  = _tonp_bool(has_higher)
    hhf_np = _tonp_bool(has_higher_fail)

    dm_lead_higher     = dm_p130_pass[hh_np]
    njets_lead_higher  = njets_p130_pass[hh_np]
    dm_matched_lead    = dm_p130_pass[~hh_np]
    njets_matched_lead = njets_p130_pass[~hh_np]
    dm_fail_higher     = dm_p130_fail[hhf_np]
    njets_fail_higher  = njets_p130_fail[hhf_np]

    # ---- summary 출력 ----
    print(f"\n{'='*60}")
    print(f"[{label}]  DR_CUT = {DR_CUT}")
    print(f"{'='*60}")
    print(f"  입력 이벤트 수                          : {len(gen_pt_sel)}")

    print(f"  ΔR < {DR_CUT} 매칭 성공                   : {ak.sum(matched_mask)}")
    print_dm_jet_table("매칭 성공", dm_matched, njets_matched)

    print(f"  매칭 실패                               : {ak.sum(~matched_mask)}")

    print(f"  gen pt>{gen_pt_cut}, |η|<{gen_eta_cut}            : {len(after_gen_sel_gen_pt)}")
    print_dm_jet_table("gen pt/η cut 통과", dm_after_cut, njets_after_cut)

    print(f"  └─ matched reco pt > {reco_pt_cut}          : {len(matched_reco_pass)}")
    print_dm_jet_table(f"matched reco pt > {reco_pt_cut}",
                       dm_p130_pass, njets_p130_pass, indent="      ")

    print(f"     ├─ leading reco > matched (unknown)  : {int(ak.sum(has_higher))}")
    print_dm_jet_table("leading reco > matched (unknown)",
                       dm_lead_higher, njets_lead_higher, indent="        ")

    print(f"     └─ matched is leading (triggered)    : {len(matched_reco_pass) - int(ak.sum(has_higher))}")
    print_dm_jet_table("matched is leading (triggered)",
                       dm_matched_lead, njets_matched_lead, indent="        ")

    print(f"  └─ matched reco pt < {reco_pt_cut}          : {len(matched_reco_fail)}")
    print_dm_jet_table(f"matched reco pt < {reco_pt_cut}",
                       dm_p130_fail, njets_p130_fail, indent="      ")

    print(f"     └─ has higher reco than matched      : {int(ak.sum(has_higher_fail))}")
    print_dm_jet_table("has higher reco than matched",
                       dm_fail_higher, njets_fail_higher, indent="        ")

    return {
        "matched_mask": matched_mask, "matched_gen_pt": matched_gen_pt,
        "matched_gen_eta": matched_gen_eta, "matched_gen_dm": matched_gen_dm,
        "matched_reco_pt": matched_reco_pt, "matched_best_dR": matched_best_dR,
        "after_gen_sel_gen_pt": after_gen_sel_gen_pt,
        "after_gen_sel_reco_pt": after_gen_sel_reco_pt,
        "after_gen_sel_dr": after_gen_sel_dr,
        "leading_reco": leading_reco,
    }


# ═══════════════════════════════════════════════════════════
# 4. gen=1 / gen=2 스캔 함수 (njets 인자 포함)
# ═══════════════════════════════════════════════════════════
def scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                 gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                 njets_1,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    gen_pt_flat  = ak.flatten(gen_pt_1)
    gen_eta_flat = ak.flatten(gen_eta_1)
    gen_phi_flat = ak.flatten(gen_phi_1)
    gen_dm_flat  = ak.flatten(gen_dm_1)
    njets_arr    = np.asarray(njets_1, dtype=int)

    results = {}
    for dr in dr_list:
        results[dr] = analyze_tau_matching(
            reco_pt_1, reco_eta_1, reco_phi_1,
            gen_pt_flat, gen_eta_flat, gen_phi_flat, gen_dm_flat,
            njets_arr,
            DR_CUT=dr, label="Gen=1",
        )
    return results


def scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                 gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                 njets_2,
                 dr_list=[0.05, 0.1, 0.2, 0.3, 0.5]):
    gen_sort_idx   = ak.argsort(gen_pt_2, axis=1, ascending=False)
    gen_pt_sorted  = gen_pt_2[gen_sort_idx]
    gen_eta_sorted = gen_eta_2[gen_sort_idx]
    gen_phi_sorted = gen_phi_2[gen_sort_idx]
    gen_dm_sorted  = gen_dm_2[gen_sort_idx]

    lead_pt  = ak.flatten(gen_pt_sorted[:, 0:1])
    lead_eta = ak.flatten(gen_eta_sorted[:, 0:1])
    lead_phi = ak.flatten(gen_phi_sorted[:, 0:1])
    lead_dm  = ak.flatten(gen_dm_sorted[:, 0:1])
    sub_pt   = ak.flatten(gen_pt_sorted[:, 1:2])
    sub_eta  = ak.flatten(gen_eta_sorted[:, 1:2])
    sub_phi  = ak.flatten(gen_phi_sorted[:, 1:2])
    sub_dm   = ak.flatten(gen_dm_sorted[:, 1:2])

    njets_arr = np.asarray(njets_2, dtype=int)

    results = {}
    for dr in dr_list:
        print(f"\n{'#'*60}\n  Gen=2  |  DR_CUT = {dr}\n{'#'*60}")

        gen_eta_bc, reco_eta_bc = ak.broadcast_arrays(lead_eta[:, np.newaxis], reco_eta_2)
        gen_phi_bc, reco_phi_bc = ak.broadcast_arrays(lead_phi[:, np.newaxis], reco_phi_2)
        deta = gen_eta_bc - reco_eta_bc
        dphi = (gen_phi_bc - reco_phi_bc + np.pi) % (2 * np.pi) - np.pi
        best_dR_lead = ak.min(np.sqrt(deta**2 + dphi**2), axis=1)

        lead_matched_mask = best_dR_lead < dr
        lead_fail_mask    = ~lead_matched_mask
        lead_matched_np   = ak.to_numpy(ak.fill_none(lead_matched_mask, False)).astype(bool)
        lead_fail_np      = ~lead_matched_np

        print(f"\n  [Leading gen matched: {ak.sum(lead_matched_mask)} / {len(lead_pt)} events]")
        res_lead = analyze_tau_matching(
            reco_pt_2[lead_matched_mask], reco_eta_2[lead_matched_mask], reco_phi_2[lead_matched_mask],
            lead_pt[lead_matched_mask], lead_eta[lead_matched_mask],
            lead_phi[lead_matched_mask], lead_dm[lead_matched_mask],
            njets_arr[lead_matched_np],
            DR_CUT=dr, label="Gen=2 Leading matched",
        )

        print(f"\n  [Leading 매칭 실패 → Subleading 시도: {ak.sum(lead_fail_mask)} events]")
        res_sub = analyze_tau_matching(
            reco_pt_2[lead_fail_mask], reco_eta_2[lead_fail_mask], reco_phi_2[lead_fail_mask],
            sub_pt[lead_fail_mask], sub_eta[lead_fail_mask],
            sub_phi[lead_fail_mask], sub_dm[lead_fail_mask],
            njets_arr[lead_fail_np],
            DR_CUT=dr, label="Gen=2 Subleading (leading failed)",
        )
        results[dr] = {"leading": res_lead, "subleading": res_sub}
    return results


# ═══════════════════════════════════════════════════════════
# 5. selection-aligned GenVisTau / GenJet (η, φ) 만들기
# ═══════════════════════════════════════════════════════════
genvis_eta_1 = genvis_eta[trigger_passed][mask2]
genvis_phi_1 = genvis_phi[trigger_passed][mask2]
genjet_eta_1 = genjet_eta[trigger_passed][mask2]
genjet_phi_1 = genjet_phi[trigger_passed][mask2]

genvis_eta_2 = genvis_eta[trigger_passed][mask3]
genvis_phi_2 = genvis_phi[trigger_passed][mask3]
genjet_eta_2 = genjet_eta[trigger_passed][mask3]
genjet_phi_2 = genjet_phi[trigger_passed][mask3]

# Gen=2 데이터 (Cell 4에서 만들었지만, 이 셀만 단독 실행해도 되도록 재정의)
gen_pt_2   = genvis_pt [trigger_passed][mask3]
gen_eta_2  = genvis_eta[trigger_passed][mask3]
gen_phi_2  = genvis_phi[trigger_passed][mask3]
gen_dm_2   = genvis_tau_dm[trigger_passed][mask3]
reco_pt_2  = tau_pt [trigger_passed][mask3]
reco_eta_2 = tau_eta[trigger_passed][mask3]
reco_phi_2 = tau_phi[trigger_passed][mask3]

# ═══════════════════════════════════════════════════════════
# 6. isolated GenJet 개수 + sanity check + 분석 실행
# ═══════════════════════════════════════════════════════════
njets_1 = count_isolated_jets(genvis_eta_1, genvis_phi_1,
                              genjet_eta_1, genjet_phi_1,
                              deltaR_threshold=0.3)
njets_2 = count_isolated_jets(genvis_eta_2, genvis_phi_2,
                              genjet_eta_2, genjet_phi_2,
                              deltaR_threshold=0.3)

assert len(njets_1) == len(gen_pt_1) == len(reco_pt_1), \
    f"gen=1 length mismatch: njets={len(njets_1)}, gen={len(gen_pt_1)}, reco={len(reco_pt_1)}"
assert len(njets_2) == len(gen_pt_2) == len(reco_pt_2), \
    f"gen=2 length mismatch: njets={len(njets_2)}, gen={len(gen_pt_2)}, reco={len(reco_pt_2)}"

res1 = scan_dr_gen1(reco_pt_1, reco_eta_1, reco_phi_1,
                    gen_pt_1,  gen_eta_1,  gen_phi_1,  gen_dm_1,
                    njets_1,
                    dr_list=[0.1])

res2 = scan_dr_gen2(reco_pt_2, reco_eta_2, reco_phi_2,
                    gen_pt_2,  gen_eta_2,  gen_phi_2,  gen_dm_2,
                    njets_2,
                    dr_list=[0.1])

NameError: name 'genvis_eta' is not defined